In [47]:
import os
import sys
import pandas as pd
import plotly.express as px
import numpy as np
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from collections import Counter
import re
import matplotlib.pyplot as plt

In [48]:
parent_directory=os.path.dirname(os.getcwd())
path=os.path.join(parent_directory,"src")
sys.path.append(parent_directory)
import src.cleaning_data as cln
BASE_DIR = os.path.dirname(os.getcwd())
proccessed_file=os.path.join(BASE_DIR,"data","processed","processed.csv")

raw_df = pd.read_csv(proccessed_file).copy()

processed_ventas_df = cln.cleaning_data_frame_by_category(raw_df,"venta",min_outliner=1000,max_outliner=300000)
processed_alquiler_df = cln.cleaning_data_frame_by_category(raw_df,"alquiler",min_outliner=10,max_outliner=1000)
processed_permutas_df = cln.cleaning_data_frame_by_category(raw_df,"permuta",min_outliner=1000,max_outliner=100000)

processed_ventas_df["Fecha"] = pd.to_datetime(processed_ventas_df["Fecha"], errors="coerce")

initial_count = len(processed_ventas_df)
processed_ventas_df = processed_ventas_df.dropna(subset=["Fecha"])

processed_ventas_df["Año"] = processed_ventas_df["Fecha"].dt.year
processed_ventas_df["Mes"] = processed_ventas_df["Fecha"].dt.month
processed_ventas_df["Año-Mes"] = processed_ventas_df["Fecha"].dt.strftime("%Y-%m")

precio_mensual = processed_ventas_df.groupby("Año-Mes", as_index=False).agg(
    Precio_Promedio=("Precio", "median"),
    Cantidad_Propiedades=("Precio", "count")
)
precio_anual = processed_ventas_df.groupby("Año", as_index=False).agg(
    Precio_Promedio=("Precio", "median"),
    Cantidad_Propiedades=("Precio", "count")
)

In [49]:
df_ventas_Casas = processed_ventas_df[processed_ventas_df["Tipo"] == "casa"].copy()
df_ventas_Apartamentos = processed_ventas_df[processed_ventas_df["Tipo"] == "apartamento"].copy()
df_permutas_Casas = processed_permutas_df[processed_permutas_df["Tipo"] == "casa"].copy()
df_permutas_Apartamentos = processed_permutas_df[processed_permutas_df["Tipo"] == "apartamento"].copy()
df_alquiler_Casas = processed_alquiler_df[processed_alquiler_df["Tipo"] == "casa"].copy()
df_alquiler_Apartamentos = processed_alquiler_df[processed_alquiler_df["Tipo"] == "apartamento"].copy()

In [50]:
def analizar_municipios(df: pd.DataFrame):

    COLOR_PRINCIPAL = "#F3D056"
    COLOR_FONDO = "#001734"
    
    NORMALIZACION_MUNICIPIOS = {
        "Plaza": "Plaza de la Revolución",
        "Habana Vieja": "La Habana Vieja"
    }
    
    df_normalizado = df.copy()
    df_normalizado["Municipio"] = df_normalizado["Municipio"].replace(NORMALIZACION_MUNICIPIOS)
    
    municipios = df_normalizado["Municipio"].unique()
    figuras = []
    
    for municipio in municipios:
        df_municipio = df_normalizado[df_normalizado["Municipio"] == municipio].copy()
        
        if df_municipio.empty:
            continue

        df_municipio["Fecha"] = pd.to_datetime(df_municipio["Fecha"])
        df_municipio["Año-Mes"] = df_municipio["Fecha"].dt.to_period("M").astype(str)
        
        precio_mensual = df_municipio.groupby("Año-Mes", as_index=False)["Precio"].median()
        
        fig = px.line(
            precio_mensual,
            x="Año-Mes",
            y="Precio",
            title=f"Evolución de Precios en {municipio}",
            labels={"Precio": "Precio mediano (USD)", "Año-Mes": "Periodo"},
            markers=True
        )
        
        fig.update_traces(
            line=dict(color=COLOR_PRINCIPAL, width=3),
            marker=dict(color=COLOR_PRINCIPAL, size=8)
        )
        
        fig.update_layout(
            template="plotly_dark",
            plot_bgcolor=COLOR_FONDO,
            paper_bgcolor=COLOR_FONDO,
            font=dict(color="white"),
            title_font=dict(color=COLOR_PRINCIPAL),
            height=500
        )
        
        figuras.append(fig)
    
    return figuras

figuras = analizar_municipios(processed_ventas_df)

for fig in figuras:
    fig.show()

In [51]:
def proporcion_ventas_permutas(ventas_df: pd.DataFrame, permutas_df: pd.DataFrame):

    COLOR_VENTAS = "#F3D056"  
    COLOR_PERMUTAS = "#2A4A6B"  
    COLOR_FONDO = "#001734" 
    
    NORMALIZACION_MUNICIPIOS = {
        "Plaza": "Plaza de la Revolución",
        "Habana Vieja": "La Habana Vieja"
    }
    
    ventas = ventas_df.copy()
    permutas = permutas_df.copy()
    
    ventas["Tipo"] = "Ventas"
    permutas["Tipo"] = "Permutas"
    
    combinado = pd.concat([ventas, permutas], ignore_index=True)
    
    combinado["Municipio"] = combinado["Municipio"].replace(NORMALIZACION_MUNICIPIOS)
    
    municipios = combinado["Municipio"].unique()
    figuras = []
    
    for municipio in municipios:
        df_municipio = combinado[combinado["Municipio"] == municipio]
        
        if df_municipio.empty:
            continue
        
        conteo_tipos = df_municipio["Tipo"].value_counts().reset_index()
        conteo_tipos.columns = ["Tipo", "Cantidad"]
        
        total = conteo_tipos["Cantidad"].sum()
        conteo_tipos["Porcentaje"] = conteo_tipos["Cantidad"] / total * 100
        
        fig = px.pie(
            conteo_tipos,
            names="Tipo",
            values="Cantidad",
            title=f"Proporción de Ventas vs Permutas en {municipio}",
            color="Tipo",
            color_discrete_map={
                "Ventas": COLOR_VENTAS,
                "Permutas": COLOR_PERMUTAS
            }
        )

        fig.update_layout(
            template="plotly_dark",
            plot_bgcolor=COLOR_FONDO,
            paper_bgcolor=COLOR_FONDO,
            font=dict(color="white"),
            title_font=dict(size=24, color=COLOR_VENTAS),
            legend_title_text="Tipo de Operación",
            annotations=[dict(
                text=f"Total: {total}",
                x=0.5, y=0.5,
                font_size=20,
                showarrow=False,
                font_color=COLOR_VENTAS
            )]
        )
        
        fig.update_traces(
            textinfo="percent+label",
            textposition="inside",
            textfont=dict(color="white", size=14),
            hovertemplate="<b>%{label}</b><br>%{value} anuncios (%{percent})"
        )
        
        figuras.append(fig)
    
    return figuras

figuras = proporcion_ventas_permutas(processed_ventas_df, processed_permutas_df)
for fig in figuras:
    fig.show()

In [52]:
def generar_grafico_precio_anual_mediana(ventas_df: pd.DataFrame):

    df = ventas_df.copy()

    df = df.dropna(subset=["Fecha"])

    if not pd.api.types.is_datetime64_any_dtype(df["Fecha"]):
        df["Fecha"] = pd.to_datetime(df["Fecha"])
    
    df["Año"] = df["Fecha"].dt.year
    df["Mes"] = df["Fecha"].dt.month
    df["Año-Mes"] = df["Fecha"].dt.strftime("%Y-%m")
    
    precio_anual = df.groupby("Año", as_index=False).agg(
        Precio_Mediano=("Precio", "median"),
        Cantidad_Propiedades=("Precio", "count")
    )
    
    fig = px.bar(
        precio_anual,
        x='Año',
        y='Precio_Mediano',
        title='Precio Mediano Anual de Propiedades',
        color='Precio_Mediano'
    )
    
    fig.update_layout(
        xaxis_title='Año',
        yaxis_title='Precio Promedio (USD)',
        template='plotly_dark',
        plot_bgcolor='#001734',
        paper_bgcolor='#001734',
        font=dict(color='white'),
        title_font=dict(color='white')
    )
    
    return fig

figura_precio_anual_mediano = generar_grafico_precio_anual_mediana(processed_ventas_df)
figura_precio_anual_mediano.show()

In [53]:
def generar_grafico_evolucion_mensual(ventas_df: pd.DataFrame):

    df = ventas_df.copy()

    df = df.dropna(subset=["Fecha"])
    
    if not pd.api.types.is_datetime64_any_dtype(df["Fecha"]):
        df["Fecha"] = pd.to_datetime(df["Fecha"])
    
    df["Año-Mes"] = df["Fecha"].dt.strftime("%Y-%m")
    
    precio_mensual = df.groupby("Año-Mes", as_index=False).agg(
        Precio_mediano=("Precio", "median"),
        Cantidad_Propiedades=("Precio", "count")
    )
    
    precio_mensual = precio_mensual.sort_values("Año-Mes")
    
    fig = px.line(
        precio_mensual,
        x='Año-Mes',
        y='Precio_mediano',
        title='Evolución Mensual de Precios de Propiedades',
        labels={'Precio_mediano': 'Precio mediano (USD)', 'Año-Mes': 'Mes'},
        markers=True
    )
    
    fig.update_layout(
        xaxis_title='Periodo',
        yaxis_title='Precio mediano (USD)',
        template='plotly_dark',
        plot_bgcolor='#001734',
        paper_bgcolor='#001734',
        font=dict(color='white'),
        title_font=dict(color='white'),
        height=500
    )

    fig.update_traces(
        line=dict(color='#F3D056', width=3),
        marker=dict(color='#F3D056', size=8)
    )
    
    return fig

figura_mensual = generar_grafico_evolucion_mensual(processed_ventas_df)
figura_mensual.show()

In [58]:
def generar_heatmap_precios_mediana(ventas_df: pd.DataFrame):

    df = ventas_df.copy()
    
    df = df.dropna(subset=["Fecha"])
    
    if not pd.api.types.is_datetime64_any_dtype(df["Fecha"]):
        df["Fecha"] = pd.to_datetime(df["Fecha"])
    
    df["Año"] = df["Fecha"].dt.year
    df["Mes"] = df["Fecha"].dt.month
    
    heatmap_data = df.pivot_table(
        index='Mes',     
        columns='Año',   
        values='Precio', 
        aggfunc='median'    
    )
    
    meses = ['Ene', 'Feb', 'Mar', 'Abr', 'May', 'Jun', 
             'Jul', 'Ago', 'Sep', 'Oct', 'Nov', 'Dic']
    heatmap_data.index = [meses[i-1] for i in heatmap_data.index]
    

    custom_scale = [
        [0.0, "#032F66"],  
        [0.2, "#355E86"], 
        [0.4, "#F0C050"],  
        [0.6, "#F3D056"], 
        [0.8, "#FFDF00"],  
        [1.0, "#FFC000"]  
    ]
    
    fig = px.imshow(
        heatmap_data,
        labels=dict(x="Año", y="Mes", color="Precio (USD)"),
        title='Precios Mediano de Propiedades por Mes y Año',
        color_continuous_scale=custom_scale
    )
    
    fig.update_layout(
        xaxis_nticks=len(heatmap_data.columns),
        yaxis_nticks=len(heatmap_data.index),
        xaxis_title='Año',
        yaxis_title='Mes',
        template='plotly_dark',
        plot_bgcolor='#001734',
        paper_bgcolor='#001734',
        font=dict(color='white'),
        title_font=dict(color='white'),
        height=600
    )

    for i, mes in enumerate(heatmap_data.index):
        for j, año in enumerate(heatmap_data.columns):
            precio = heatmap_data.loc[mes, año]
            if not pd.isnull(precio):
                fig.add_annotation(
                    x=año,
                    y=mes,
                    text=f"{precio:,.0f}",
                    showarrow=False,
                    font=dict(
                        size=10,
                        color='white' if precio > heatmap_data.values.mean() else 'black'
                    )
                )
    
    return fig

heatmap_fig = generar_heatmap_precios_mediana(processed_ventas_df)
heatmap_fig.show()

In [55]:
def analizar_amenidades_ventas(ventas_df: pd.DataFrame, top_n=15):

    COLOR_PRINCIPAL = '#F3D056'
    COLOR_FONDO = '#001734'
    COLOR_SECUNDARIO = '#2A4A6B'
    
    amenities_counter = Counter()
    
    normalization_map = {
        'parqueo': 'Estacionamiento',
        'garaje': 'Estacionamiento',
        'cochera': 'Estacionamiento',
        'aire': 'Aire Acondicionado',
        'balcon': 'Balcón',
        "telefono fijo": "Teléfono fijo",
        "teléfono": "Teléfono fijo",
        "tanque instalado": "Tanque elevado",
        "placa libre": "Terraza",
        "tanques propios": "Tanque elevado",
        "split": "Aire Acondicionado"
    }

    def normalize_amenity(amenity):
        amenity = amenity.strip().lower()
        
        for key, value in normalization_map.items():
            if key in amenity:
                return value
            
        return amenity.capitalize()
    

    for amenities in ventas_df['Amenidades'].dropna():
        if isinstance(amenities, list):
            normalized = [normalize_amenity(a) for a in amenities]
            amenities_counter.update(normalized)
        
        elif isinstance(amenities, str):
            split_amenities = re.split(r'[,;|]', amenities)
            normalized = [normalize_amenity(a.strip()) for a in split_amenities if a.strip()]
            amenities_counter.update(normalized)
    
    top_amenities = amenities_counter.most_common(top_n)
    df_amenities = pd.DataFrame(top_amenities, columns=['Amenidad', 'Cantidad'])
    

    fig_bar = px.bar(
        df_amenities.sort_values('Cantidad', ascending=True),
        x='Cantidad',
        y='Amenidad',
        orientation='h',
        title='Amenidades más Comunes en Propiedades en Ventas',
        labels={'Cantidad': 'Número de Propiedades', 'Amenidad': ''},
        color='Cantidad',
        color_continuous_scale=[(0, COLOR_SECUNDARIO), (1, COLOR_PRINCIPAL)]
    )
    
    fig_bar.update_layout(
        template='plotly_dark',
        plot_bgcolor=COLOR_FONDO,
        paper_bgcolor=COLOR_FONDO,
        font=dict(color='white'),
        title_font=dict(size=24, color=COLOR_PRINCIPAL),
        xaxis_title='Número de Propiedades',
        yaxis_title='Amenidad',
        height=600,
        showlegend=False,
        margin=dict(l=100, r=50, t=80, b=50)
    )
    
    fig_bar.update_traces(
        marker_line_color='white',
        marker_line_width=1
    )
    
    fig_bar.update_coloraxes(showscale=False)
    
    fig_pie = px.pie(
        df_amenities,
        names='Amenidad',
        values='Cantidad',
        title='Distribución de Amenidades en Ventas',
        hole=0.2
    )
    

    fig_pie.update_layout(
        template='plotly_dark',
        plot_bgcolor=COLOR_FONDO,
        paper_bgcolor=COLOR_FONDO,
        font=dict(color='white'),
        title_font=dict(size=20, color=COLOR_PRINCIPAL),
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=-0.3,
            xanchor="center",
            x=0.5
        )
    )
    
    fig_pie.update_traces(
        textposition='inside',
        textinfo='percent+label',
        hovertemplate='<b>%{label}</b><br>%{value} propiedades (%{percent})'
    )
    
    fig_final = make_subplots(
        rows=1, cols=2,
        specs=[[{"type": "bar"}, {"type": "pie"}]],
        subplot_titles=('Top Amenidades', 'Distribución Porcentual'),
        horizontal_spacing=0.1
    )

    for trace in fig_bar.data:
        fig_final.add_trace(trace, row=1, col=1)
    
    fig_final.add_trace(fig_pie.data[0], row=1, col=2)
    
    fig_final.update_layout(
        title_text='Análisis de Amenidades en Propiedades en Ventas',
        title_font=dict(size=24, color=COLOR_PRINCIPAL),
        template='plotly_dark',
        plot_bgcolor=COLOR_FONDO,
        paper_bgcolor=COLOR_FONDO,
        font=dict(color='white'),
        height=500,
        showlegend=False
    )
    
    fig_final.update_annotations(font_color=COLOR_PRINCIPAL)
    
    fig_final.show()
    
    return df_amenities, amenities_counter

df_top_amenidades, contador_amenidades = analizar_amenidades_ventas(processed_ventas_df)

In [56]:
def analizar_amenidades_por_tipo(df: pd.DataFrame, tipo_col='Tipo', top_n=10):
    
    df_casas = df[df[tipo_col].str.contains('Casa|House', case=False, na=False)]
    df_aptos = df[df[tipo_col].str.contains('Apartamento|Apartment|Apto', case=False, na=False)]
    
    # Diccionario de normalización
    normalization_map = {
        'parqueo': 'Estacionamiento',
        'garaje': 'Estacionamiento',
        'cochera': 'Estacionamiento',
        'aire': 'Aire Acondicionado',
        'balcon': 'Balcón',
        "telefono fijo": "Teléfono fijo",
        "teléfono": "Teléfono fijo",
        "tanque instalado": "Tanque elevado",
        "placa libre": "Terraza",
        "tanques propios": "Tanque elevado",
        "split": "Aire Acondicionado"
    }
    
    def normalize_amenity(amenity):
        amenity = str(amenity).strip().lower()
        for key, value in normalization_map.items():
            if key in amenity:
                return value
        return amenity.capitalize()
    
    def procesar_amenidades(df_amenities):
        counter = Counter()
        for amenities in df_amenities['Amenidades'].dropna():
            if isinstance(amenities, list):
                normalized = [normalize_amenity(a) for a in amenities]
                counter.update(normalized)
            elif isinstance(amenities, str):
                split_amenities = re.split(r'[,;|]', amenities)
                normalized = [normalize_amenity(a) for a in split_amenities if a.strip()]
                counter.update(normalized)
        return counter
    

    contador_casas = procesar_amenidades(df_casas)
    df_casas_count = pd.DataFrame(contador_casas.most_common(top_n), 
                                  columns=['Amenidad', 'Casas'])
    
    contador_aptos = procesar_amenidades(df_aptos)
    df_aptos_count = pd.DataFrame(contador_aptos.most_common(top_n), 
                                  columns=['Amenidad', 'Apartamentos'])
    
    df_comparativo = pd.merge(df_casas_count, df_aptos_count, 
                              on='Amenidad', how='outer').fillna(0)
    
    df_comparativo['Total'] = df_comparativo['Casas'] + df_comparativo['Apartamentos']
    df_comparativo = df_comparativo.sort_values('Total', ascending=False).head(top_n)
    
    COLOR_CASAS = '#F3D056'  
    COLOR_APTOS = '#2A4A6B'  
    COLOR_FONDO = '#001734'  
    
    fig = px.bar(
        df_comparativo,
        x='Amenidad',
        y=['Casas', 'Apartamentos'],
        title='Amenidades Más Comunes por Tipo de Propiedad',
        labels={'value': 'Número de Propiedades', 'Amenidad': 'Amenidad'},
        barmode='group',
        color_discrete_sequence=[COLOR_CASAS, COLOR_APTOS]
    )
    
    fig.update_layout(
        template='plotly_dark',
        plot_bgcolor=COLOR_FONDO,
        paper_bgcolor=COLOR_FONDO,
        font=dict(color='white'),
        title_font=dict(size=24, color=COLOR_CASAS),
        legend_title_text='Tipo de Propiedad',
        xaxis_tickangle=-45,
        height=600,
        margin=dict(l=50, r=50, t=100, b=150)
    )
    
    fig.update_traces(
        hovertemplate='<b>%{x}</b><br>Tipo: %{meta[0]}<br>Propiedades: %{y}',
        marker_line_color='white',
        marker_line_width=1,
        meta=[['Casas']*len(df_comparativo), ['Apartamentos']*len(df_comparativo)]
    )
    
    fig.show()
    return df_comparativo

df_amenidades_tipo = analizar_amenidades_por_tipo(
    processed_ventas_df, 
    tipo_col='Tipo', 
    top_n=12
)

In [57]:
def valor_amenidades_simplificado(df: pd.DataFrame, top_n=15):
    
    normalization_map = {
        'parqueo': 'Estacionamiento',
        'garaje': 'Estacionamiento',
        'cochera': 'Estacionamiento',
        'aire': 'Aire Acondicionado',
        'balcon': 'Balcón',
        "telefono fijo": "Teléfono fijo",
        "teléfono": "Teléfono fijo",
        "tanque instalado": "Tanque elevado",
        "placa libre": "Terraza",
        "tanques propios": "Tanque elevado",
        "split": "Aire Acondicionado"
    }
    
    def normalize_amenity(amenity):
        amenity = str(amenity).strip().lower()
        for key, value in normalization_map.items():
            if key in amenity:
                return value
        return amenity.capitalize()
    
    def normalize_amenities(amenities):
        if isinstance(amenities, list):
            return [normalize_amenity(a) for a in amenities]
        elif isinstance(amenities, str):
            split_amenities = re.split(r'[,;|]', amenities)
            return [normalize_amenity(a.strip()) for a in split_amenities if a.strip()]
        return []
    

    df_normalized = df.copy()
    df_normalized['Amenidades_Normalizadas'] = df_normalized['Amenidades'].apply(normalize_amenities)
    
    all_amenities = []
    for amenities in df_normalized['Amenidades_Normalizadas']:
        all_amenities.extend(amenities)
    
    if not all_amenities:
        print("No se encontraron amenidades para analizar")
        return None
    
    amenities_counts = pd.Series(all_amenities).value_counts().head(top_n)
    top_amenidades = amenities_counts.index.tolist()
    
    resultados = []
    for amenidad in top_amenidades:
        mask = df_normalized['Amenidades_Normalizadas'].apply(
            lambda x: amenidad in x
        )
        
        propiedades_con = df_normalized[mask]
        propiedades_sin = df_normalized[~mask]
        
        if len(propiedades_con) > 5 and len(propiedades_sin) > 5:
            precio_con = propiedades_con['Precio'].median()
            precio_sin = propiedades_sin['Precio'].median()
            
            if precio_sin > 0: 
                incremento = ((precio_con - precio_sin) / precio_sin) * 100
                resultados.append({
                    'Amenidad': amenidad,
                    'Incremento': incremento,
                    'Propiedades_Con': len(propiedades_con),
                    'Precio_Con': precio_con,
                    'Precio_Sin': precio_sin
                })

    if not resultados:
        print("No se pudo calcular el impacto para ninguna amenidad")
        return None
    
    df_resultados = pd.DataFrame(resultados).sort_values('Incremento', ascending=False)
    

    COLOR_POSITIVO = '#F3D056'  
    COLOR_NEGATIVO = '#2A4A6B'  
    COLOR_FONDO = '#001734'   
    
    fig = px.bar(
        df_resultados,
        x='Amenidad',
        y='Incremento',
        title='Impacto de Amenidades en Precios de Venta',
        labels={'Incremento': 'Incremento Porcentual (%)', 'Amenidad': 'Amenidad'},
        text='Incremento',
        color='Incremento',
        color_continuous_scale=[(0, COLOR_NEGATIVO), (0.5, 'gray'), (1, COLOR_POSITIVO)],
        range_color=[min(df_resultados['Incremento']) - 10, max(df_resultados['Incremento']) + 10]
    )
    

    fig.update_layout(
        template='plotly_dark',
        plot_bgcolor=COLOR_FONDO,
        paper_bgcolor=COLOR_FONDO,
        font=dict(color='white'),
        title_font=dict(size=22, color=COLOR_POSITIVO),
        xaxis_tickangle=-45,
        height=600,
        coloraxis_showscale=False
    )
    

    fig.update_traces(
        texttemplate='%{text:.1f}%',
        textposition='outside',
        marker_line_color='white',
        marker_line_width=1,
        hovertemplate=(
            '<b>%{x}</b><br>'
            'Incremento: %{y:.1f}%<br>'
            'Propiedades: %{customdata[0]}<br>'
            'Precio con: $%{customdata[1]:,.0f}<br>'
            'Precio sin: $%{customdata[2]:,.0f}'
        ),
        customdata=df_resultados[['Propiedades_Con', 'Precio_Con', 'Precio_Sin']]
    )
    
    fig.show()
    return df_resultados

df_impacto = valor_amenidades_simplificado(processed_ventas_df)